In [2]:
# ===============================
# 1. Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

# ===============================
# 2. Load your dataset
# ===============================
# Replace with your actual dataset path
data = pd.read_csv("./data/heart_2020_cleaned.csv")


X = data.drop('HeartDisease', axis=1)
y = data['HeartDisease']

# ===============================
# 3. Train/test split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===============================
# 4. Preprocessing
# ===============================
# Identify numeric and categorical features
X_num = X_train.select_dtypes(include='number')
X_cat = X_train.select_dtypes(exclude='number')

# Categorical columns
cat_cols = X_cat.columns

# Numeric columns (all others)
num_cols = X_num.columns

# Numeric pipeline
num_pipe = Pipeline([
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# ===============================
# 5. Logistic Regression Classifier
# ===============================
log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
    C=0.1,
    penalty="l2",
    solver="liblinear"
)

pipe_clf = Pipeline([
    ("prep", preprocessor),
    ("model", log_reg)
])

pipe_clf.fit(X_train, y_train)

# Predicted probabilities (risk scores)
train_probs = pipe_clf.predict_proba(X_train)[:, 1]
test_probs = pipe_clf.predict_proba(X_test)[:, 1]

# ===============================
# 6. Regression Model to Predict Risk Scores
# ===============================
y_train_reg = train_probs
y_test_reg = test_probs

reg_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

pipe_reg = Pipeline([
    ("prep", preprocessor),
    ("reg", reg_model)
])

pipe_reg.fit(X_train, y_train_reg)

# Predict risk scores on test set
risk_pred = pipe_reg.predict(X_test)

# ===============================
# 7. Evaluate Regression
# ===============================
print("MAE:", mean_absolute_error(y_test_reg, risk_pred))
print("R²:", r2_score(y_test_reg, risk_pred))
print("Correlation:", np.corrcoef(y_test_reg, risk_pred)[0,1])

# ===============================
# 8. Convert to Risk Percentage & Category
# ===============================
# Convert probability to percentage
risk_percentage = risk_pred * 100

# Define risk categories
def get_risk_category(prob, threshold=0.19):
    if prob < threshold:
        return "Low"
    elif prob < 0.5:
        return "Moderate"
    elif prob < 0.8:
        return "High"
    else:
        return "Very High"

risk_category = [get_risk_category(p, threshold=0.19) for p in risk_pred]

# Combine into a DataFrame for display
risk_results = pd.DataFrame({
    "Predicted Probability": risk_pred,
    "Risk Percentage": risk_percentage,
    "Risk Category": risk_category
})

# Show first few results
risk_results.head()


MAE: 0.05709478120761218
R²: 0.9043547502405335
Correlation: 0.9510077703242301


,Predicted Probability,Risk Percentage,Risk Category
0,0.577631,57.763136,High
1,0.545079,54.507918,High
2,0.101797,10.179681,Low
3,0.818837,81.883745,Very High
4,0.169658,16.965776,Low
